# Colab 01 — Ablation de Resolução (640×640 vs 1024×1024 — Modelo V1) e Expansão V2 (`Pacote 1 + Pacote 2`)

Este notebook reproduz integralmente em GPU de nuvem (Google Colab T4 / L4 / A100) as **Etapas 1 e 2** do treinamento supervisionado do **ArborGrid Vision**:
1. Descompacta `arborgrid_v3_gold_dataset.zip` (gerado e verificado pelo notebook local `01_coleta_geosampa_e_prelabel_generico.ipynb`).
2. Treina o **Modelo Intermediário V1 (`arborgrid_segformer_b2_1024_gold.pt`)** em `1024×1024` sobre as **68 imagens do `pacote_1`** (54 treino / 14 validação *hold-out*), superando o gargalo de sub-amostragem (`stride 4`) da resolução `640×640`.
3. Treina o **Modelo V2 Expandido (`arborgrid_segformer_b2_1024_v2_full.pt`)** em `1024×1024` combinando `pacote_1 + pacote_2` (**134 imagens**).
4. Exporta os relatórios JSON (`etapa1_ablation_1024px_metrics.json` e `etapa2_v2_full_132imgs_metrics.json`) consumidos localmente pelo notebook `02_ablation_v1_resolucao_e_expansao_v2.ipynb`.

In [ ]:
# 1. Verificação de GPU e instalação de dependências no Google Colab
!nvidia-smi
!pip install -q transformers accelerate opencv-python-headless

In [ ]:
import json
import zipfile
from pathlib import Path
import cv2
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import SegformerForSemanticSegmentation

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo de treinamento:", DEVICE)

ZIP_PATH = Path("arborgrid_v3_gold_dataset.zip")
WORK_DIR = Path("/content/arborgrid_dataset")
if ZIP_PATH.exists():
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(WORK_DIR)
    print("Dataset extraído em:", WORK_DIR)

In [ ]:
# 2. Dataset Multi-Label Sigmoid (3 canais: 0=copa, 1=fiacao, 2=poste) em 1024x1024
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def json_to_3ch_mask(json_path: Path, h: int = 1024, w: int = 1024) -> np.ndarray:
    mask = np.zeros((3, h, w), dtype=np.float32)
    data = json.loads(json_path.read_text(encoding="utf-8"))
    orig_w = float(data.get("imageWidth") or 640.0)
    orig_h = float(data.get("imageHeight") or 640.0)
    sx, sy = w / orig_w, h / orig_h
    for sh in data.get("shapes", []):
        lbl = sh.get("label", "").lower().strip()
        cid = 0 if "copa" in lbl else 1 if ("fio" in lbl or "fiacao" in lbl) else 2 if "poste" in lbl else None
        pts = sh.get("points", [])
        if cid is not None and len(pts) >= 3:
            scaled = np.round(np.array([[p[0] * sx, p[1] * sy] for p in pts])).astype(np.int32)
            cv2.fillPoly(mask[cid], [scaled], 1.0)
    # Espessura mínima de 3px na fiação em 1024x1024
    if mask[1].sum() > 0:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        mask[1] = cv2.dilate(mask[1], k, iterations=1)
    return mask

class ArborGridSegDataset(Dataset):
    def __init__(self, pairs: list[tuple[Path, Path]], res: int = 1024, augment: bool = False):
        self.pairs = pairs
        self.res = res
        self.augment = augment

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_p, json_p = self.pairs[idx]
        img = cv2.imread(str(img_p))
        img = cv2.resize(img, (self.res, self.res), interpolation=cv2.INTER_LINEAR)
        mask = json_to_3ch_mask(json_p, self.res, self.res)
        if self.augment and np.random.rand() > 0.5:
            img = cv2.flip(img, 1)
            mask = mask[:, :, ::-1].copy()
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        norm = ((rgb - IMAGENET_MEAN) / IMAGENET_STD).transpose(2, 0, 1)
        return torch.from_numpy(norm), torch.from_numpy(mask)

In [ ]:
# 3. Função de Perda Híbrida (Weighted BCE + Soft Dice Loss para Fiação Fina) e Loop de Treino
def hybrid_bce_dice_loss(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    up = F.interpolate(logits, size=targets.shape[-2:], mode="bilinear", align_corners=False)
    pos_weight = torch.tensor([1.5, 6.0, 3.0], device=logits.device).view(1, 3, 1, 1)
    bce = F.binary_cross_entropy_with_logits(up, targets, pos_weight=pos_weight)
    probs = torch.sigmoid(up)
    inter = (probs * targets).sum(dim=(2, 3))
    union = probs.sum(dim=(2, 3)) + targets.sum(dim=(2, 3))
    dice = 1.0 - ((2.0 * inter + 1.0) / (union + 1.0)).mean()
    return 0.5 * bce + 0.5 * dice

def train_segformer_1024(train_pairs, val_pairs, out_ckpt: str, epochs: int = 45, lr: float = 1.5e-4):
    model = SegformerForSemanticSegmentation.from_pretrained(
        "nvidia/segformer-b2-finetuned-ade-512-512", num_labels=3, ignore_mismatched_sizes=True
    ).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.amp.GradScaler("cuda")

    train_loader = DataLoader(ArborGridSegDataset(train_pairs, 1024, True), batch_size=4, shuffle=True)
    val_loader = DataLoader(ArborGridSegDataset(val_pairs, 1024, False), batch_size=2, shuffle=False)

    best_wire_iou = 0.0
    for ep in range(1, epochs + 1):
        model.train()
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            with torch.amp.autocast("cuda"):
                loss = hybrid_bce_dice_loss(model(pixel_values=x).logits, y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
        sched.step()

        model.eval()
        inter_w, union_w = 0.0, 0.0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                logits = model(pixel_values=x).logits
                up = F.interpolate(logits, size=y.shape[-2:], mode="bilinear", align_corners=False)
                pred_w = (torch.sigmoid(up)[:, 1] >= 0.35)
                gt_w = y[:, 1] > 0.5
                inter_w += float((pred_w & gt_w).sum())
                union_w += float((pred_w | gt_w).sum())
        wire_iou = inter_w / max(union_w, 1.0)
        if wire_iou >= best_wire_iou:
            best_wire_iou = wire_iou
            torch.save(model.state_dict(), out_ckpt)
        if ep % 5 == 0 or ep == epochs:
            print(f"[Ep {ep:02d}/{epochs}] Val Fiação IoU: {wire_iou:.4f} (Best: {best_wire_iou:.4f})")
    print(f"Melhor checkpoint salvo em: {out_ckpt} (Fiação IoU = {best_wire_iou:.4f})")